# SpaMosaic:三数据集指标评估

`DATASET` 切换三个马赛克数据集(结构均为 S1=RNA+ADT、S2=仅RNA、S3=仅ADT),训练后计算 **ARI、F1、PAS、CHAOS、iLISI**。

参数全部沿用官方教程:

| DATASET | input_key | T | 聚类 | 参考 |
|---|---|---|---|---|
| Simulations | X_pca_be | 0.1 | kmeans, 5 | demo/Simulation_datasets.ipynb |
| Lymph-S1-S2-S3 | dimred_bc | 0.01 | mclust, 10 | demo/Human_lymph.ipynb |
| Tonsil-S1-S2-S3 | dimred_bc | 0.01 | mclust, 6 | docs/.../mosaic/Human_tonsil_rna+adt.ipynb |

说明:
- Tonsil 官方教程聚 6 类,而真值 `lab` 只有 4 类,ARI 仍直接与 lab 比。
- Tonsil 有少量无标签细胞(lab 为 NaN),算 ARI/F1 时剔除;PAS/CHAOS/iLISI 不需要真值,用全部细胞。
- F1(LabTransfer)只对模态不重叠的切片对计算,即 slice2–slice3。

In [ ]:
import os
import sys
from os.path import join, abspath, exists

REPO_ROOT = abspath('.')
assert exists(join(REPO_ROOT, 'spamosaic')), f'请在仓库根目录启动 notebook,当前目录: {REPO_ROOT}'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # CUDA >= 10.2,须在 import torch 前设置

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

import spamosaic.utils as utls
from spamosaic.train_utils import set_seeds
from spamosaic.framework import SpaMosaic
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess

## 0. 配置

In [ ]:
DATASET = 'Simulations'                # 'Simulations' / 'Lymph-S1-S2-S3' / 'Tonsil-S1-S2-S3'
SIM_SETTING = 'rna-0.1-12-adt-2.1'     # 仅 Simulations 用
SIM_REP = '1'                          # 仅 Simulations 用
DEVICE = 'cuda:0'
SEED = 1234

PARAMS = {
    'Simulations':     dict(input_key='X_pca_be', T=0.1,  n_cluster=5,  cluster_algo='kmeans', gt_key='gt',  spot_size=200),
    'Lymph-S1-S2-S3':  dict(input_key='dimred_bc', T=0.01, n_cluster=10, cluster_algo='mclust', gt_key='lab', spot_size=70),
    'Tonsil-S1-S2-S3': dict(input_key='dimred_bc', T=0.01, n_cluster=6,  cluster_algo='mclust', gt_key='lab', spot_size=70),
}
P = PARAMS[DATASET]

set_seeds(SEED)
OUT_DIR = join(REPO_ROOT, 'examples/out', DATASET)
os.makedirs(OUT_DIR, exist_ok=True)
sc.settings.figdir = OUT_DIR

## 1. 加载数据

In [ ]:
if DATASET == 'Simulations':
    data_dir = join(REPO_ROOT, 'data/processed/Simulations', SIM_SETTING, SIM_REP)
    LAYOUT = {'rna': ['ad_rna1', 'ad_rna2', None], 'adt': ['ad_adt1', None, 'ad_adt2']}
else:
    data_dir = join(REPO_ROOT, 'data/processed', DATASET)
    LAYOUT = {'rna': ['S1/ad_rna', 'S2/ad_rna', None], 'adt': ['S1/ad_adt', None, 'S3/ad_adt']}

raw = {}
for names in LAYOUT.values():
    for n in names:
        if n is not None and n not in raw:
            raw[n] = sc.read_h5ad(join(data_dir, n + '.h5ad'))
            print(f'loaded {n}: {raw[n].shape}')

N_SEC = 3
# 每个切片取第一个存在的模态作代表(取 gt 用)
SEC_REP = [next(LAYOUT[m][i] for m in LAYOUT if LAYOUT[m][i] is not None) for i in range(N_SEC)]

In [ ]:
# Simulations 原始文件无 src 且 X 不是 counts,按官方教程补上
if DATASET == 'Simulations':
    src_labels = ['mult', 'rna', 'adt']
    for names in LAYOUT.values():
        for i, n in enumerate(names):
            if n is not None:
                raw[n].obs['src'] = src_labels[i]
                raw[n].X = raw[n].layers['counts'].copy()

input_dict = {m: [raw[n] if n is not None else None for n in names] for m, names in LAYOUT.items()}

## 2. 预处理

In [ ]:
if DATASET == 'Simulations':
    RNA_preprocess(input_dict['rna'], batch_corr=True, favor='scanpy', n_hvg=None,
                   lognorm=False, scale=True, batch_key='src', key=P['input_key'])
    ADT_preprocess(input_dict['adt'], batch_corr=True, favor='scanpy',
                   lognorm=False, scale=True, batch_key='src', key=P['input_key'])
else:
    RNA_preprocess(input_dict['rna'], batch_corr=True, favor='scanpy', n_hvg=5000,
                   batch_key='src', key=P['input_key'])
    ADT_preprocess(input_dict['adt'], batch_corr=True, batch_key='src', key=P['input_key'])

for m, ads in input_dict.items():
    print(m, [ad.obsm[P['input_key']].shape if ad is not None else None for ad in ads])

## 3. 训练

In [ ]:
model = SpaMosaic(
    modBatch_dict=input_dict, input_key=P['input_key'],
    batch_key='src', intra_knns=10, inter_knn_base=10, w_g=0.8,
    seed=SEED, device=DEVICE,
)
model.train(net='wlgcn', lr=0.01, T=P['T'], n_epochs=100)

## 4. 推理嵌入 + 聚类

In [ ]:
ad_embs = model.infer_emb(input_dict, emb_key='emb', final_latent_key='merged_emb')
ad_mosaic = sc.concat(ad_embs)
ad_mosaic = utls.get_umap(ad_mosaic, use_reps=['merged_emb'])

# mclust 需要 R + rpy2,失败自动退回 kmeans
ad_mosaic = utls.clustering(ad_mosaic, n_cluster=P['n_cluster'], used_obsm='merged_emb',
                            algo=P['cluster_algo'], key='cluster')
utls.split_adata_ob(ad_embs, ad_mosaic, 'obs', 'cluster')

utls.plot_basis(ad_mosaic, basis='merged_emb_umap', color=['src', 'cluster'], save='_umap.png')
for i, ad in enumerate(ad_embs):
    utls.plot_basis(ad, 'spatial', 'cluster', s=P['spot_size'], save=f'_spatial_slice{i+1}.png')

## 5. 指标:ARI / F1 / PAS / CHAOS / iLISI

- ARI:与真值比(Simulations 用 `gt`,Lymph/Tonsil 用 `lab`),逐切片 + 整体
- F1:LabTransfer,仅模态不重叠切片对(slice2–slice3)
- PAS / CHAOS:逐切片,聚类标签 + 空间坐标
- iLISI:合并嵌入按 src,整体

In [ ]:
# 官方评估代码不是包,按目录加入 sys.path
METRICS_DIR = join(REPO_ROOT, 'reproduce/code/compute_metrics')
if METRICS_DIR not in sys.path:
    sys.path.insert(0, METRICS_DIR)
import metrics as repro_metrics
from sklearn.metrics import adjusted_rand_score

SEC_NAMES = [f'slice{i+1}' for i in range(N_SEC)]
gt_key = P['gt_key']

gts, preds = [], []
for i, ad in enumerate(ad_embs):
    gt = raw[SEC_REP[i]].obs.loc[ad.obs_names, gt_key].astype(str)
    keep = gt != 'nan'  # Tonsil 有无标签细胞
    if keep.sum() < len(gt):
        print(f'{SEC_NAMES[i]}: 剔除 {len(gt) - keep.sum()} 个无标签细胞')
    gts.append(gt[keep])
    preds.append(ad.obs['cluster'].astype(str)[keep])

scores = {}
scores['ARI'] = {'all': adjusted_rand_score(pd.concat(gts), pd.concat(preds))}
scores['ARI'].update({n: adjusted_rand_score(g, p) for n, g, p in zip(SEC_NAMES, gts, preds)})

SEC_MODS = [{m for m in LAYOUT if LAYOUT[m][i] is not None} for i in range(N_SEC)]
scores['F1'] = {}
for i in range(N_SEC):
    for j in range(i + 1, N_SEC):
        if SEC_MODS[i] & SEC_MODS[j]:
            continue
        pair = []
        for k in (i, j):
            adx = sc.AnnData(np.zeros((len(gts[k]), 2)),
                             obs=pd.DataFrame({'gt': gts[k].values}, index=gts[k].index))
            adx.obsm['emb'] = ad_embs[k][gts[k].index].obsm['merged_emb']
            pair.append(adx)
        scores['F1'][f'{SEC_NAMES[i]}-{SEC_NAMES[j]}'] = repro_metrics.LabTransfer(*pair, 'emb', 'gt', knn=10)

for metric in ['ARI', 'F1']:
    print(metric, {k: round(v, 4) for k, v in scores[metric].items()})

In [ ]:
# PAS/CHAOS 逐切片(不需要真值,全部细胞)
scores['PAS'] = {n: repro_metrics.compute_PAS2(ad, 'cluster', spatial_key='spatial') for n, ad in zip(SEC_NAMES, ad_embs)}
scores['CHAOS'] = {n: repro_metrics.compute_CHAOS2(ad, 'cluster', spatial_key='spatial') for n, ad in zip(SEC_NAMES, ad_embs)}

# iLISI 整体(官方函数会原地改 obs,传副本)
ad_lisi = sc.AnnData(np.zeros((ad_mosaic.n_obs, 2)), obs=ad_mosaic.obs[['src']].copy())
ad_lisi.obsm['emb'] = ad_mosaic.obsm['merged_emb']
scores['iLISI'] = {'all': repro_metrics.iLISI(ad_lisi, 'src', 'emb')}

for metric in ['PAS', 'CHAOS', 'iLISI']:
    print(metric, {k: round(v, 4) for k, v in scores[metric].items()})

In [ ]:
rows = ['all'] + SEC_NAMES + list(scores['F1'])
df_scores = pd.DataFrame(scores).reindex(rows)
display(df_scores.round(4))
df_scores.to_csv(join(OUT_DIR, 'metrics.csv'))
print('已保存到', join(OUT_DIR, 'metrics.csv'))